In [53]:
# Libraries needed for import
import pandas as pd
import numpy as np

In [63]:
# Create an empty list to store the data frames
dataframes = []

# Load the dataset
data_folder = "fall2026_L"

# Make a list of all the files
all_files = [
    "ids_0.csv",
    "ids_1.csv",
    "ids_2.csv",
    "ids_3.json",
    "ids_4.json",
    "ids_5.parquet",
    "ids_6.parquet",
    "ids_7.json",
    "ids_8.parquet"
]

# Read each file
for file in all_files:

    if file.endswith(".csv"):
        df_temp = pd.read_csv(data_folder + "/" + file)

    elif file.endswith(".json"):
        df_temp = pd.read_json(data_folder + "/" + file, lines=True)

    elif file.endswith(".parquet"):
        df_temp = pd.read_parquet(data_folder + "/" + file)

    # Add the original filename
    df_temp["source_file"] = file

    # Store the DataFrame
    dataframes.append(df_temp)

    print(file, ":", df_temp.shape)

# Combine all data frames
df = pd.concat(dataframes, ignore_index=True)

# Display dimensions
print("\nCombined dataset shape:", df.shape)

ids_0.csv : (1001, 80)
ids_1.csv : (1001, 80)
ids_2.csv : (1001, 80)
ids_3.json : (1001, 80)
ids_4.json : (1001, 80)
ids_5.parquet : (15001, 80)
ids_6.parquet : (5001, 80)
ids_7.json : (9000, 80)
ids_8.parquet : (10293, 80)

Combined dataset shape: (44300, 80)


In [87]:
# Remove leading and trailing spaces from column names
df.columns = df.columns.str.strip()

In [88]:
# Show the first 20 rows of the dataset
df.head(20)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,source_file
0,443,300491,7,5,376,3508,201,0,53.714286,78.686539,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,ids_0.csv
1,37218,21,2,1,0,0,0,0,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,ids_0.csv
2,53,176,2,2,148,212,74,74,74.000000,0.000000,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,ids_0.csv
3,53,218,2,2,108,330,54,54,54.000000,0.000000,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,ids_0.csv
4,53,30753,2,2,70,126,35,35,35.000000,0.000000,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,ids_0.csv
5,443,125676,9,5,506,3743,204,0,56.222222,79.452467,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,ids_0.csv
6,123,16306,1,1,48,48,48,48,48.000000,0.000000,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,ids_0.csv
7,56434,2242,2,0,12,0,6,6,6.000000,0.000000,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,ids_0.csv
8,443,5659933,9,6,380,4091,187,0,42.222222,75.935462,...,256058.0,0.0,256058,256058,5403872.0,0.0,5403872,5403872,BENIGN,ids_0.csv
9,39080,214,3,1,31,6,31,0,10.333333,17.897858,...,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,ids_0.csv


In [41]:
# Check the data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 44300 entries, 0 to 44299
Data columns (total 80 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Destination Port             44300 non-null  int64  
 1   Flow Duration                44300 non-null  int64  
 2   Total Fwd Packets            44300 non-null  int64  
 3   Total Backward Packets       44300 non-null  int64  
 4   Total Length of Fwd Packets  44300 non-null  int64  
 5   Total Length of Bwd Packets  44300 non-null  int64  
 6   Fwd Packet Length Max        44300 non-null  int64  
 7   Fwd Packet Length Min        44300 non-null  int64  
 8   Fwd Packet Length Mean       44300 non-null  float64
 9   Fwd Packet Length Std        44300 non-null  float64
 10  Bwd Packet Length Max        44300 non-null  int64  
 11  Bwd Packet Length Min        44300 non-null  int64  
 12  Bwd Packet Length Mean       44300 non-null  float64
 13  Bwd Packet Length Std      

In [69]:
# Ignore source_file while checking for duplicates
columns_to_check = df.columns.drop("source_file")

# Count the duplicate number of records
print("Duplicate rows:", df.duplicated(subset=columns_to_check).sum())

# Count all rows that are part of a duplicate group, including the first occurrence
print("Rows included in duplicate groups:", df.duplicated(subset=columns_to_check, keep=False).sum())

Duplicate rows: 6992
Rows included in duplicate groups: 7332


In [70]:
# Remove duplicate records and keep the first occurrence
df_clean = df.drop_duplicates(subset=columns_to_check).copy()

# Display results
print("Original dataset:", df.shape)
print("Cleaned dataset:", df_clean.shape)
print("Duplicates remaining:", df_clean.duplicated(subset=columns_to_check).sum())

Original dataset: (44300, 80)
Cleaned dataset: (37308, 80)
Duplicates remaining: 0


In [71]:
# Check for missing values
missing_values = df_clean.isna().sum()

# Display only columns that contain missing values
print(missing_values[missing_values > 0])

Flow Bytes/s      7
Flow Packets/s    3
dtype: int64


In [72]:
# Identify rows containing missing values
missing_rows = df_clean[df_clean.isna().any(axis=1)]

# Display the number of affected rows
print("Rows with missing values:", len(missing_rows))

# Display the affected classes
print("\nClasses affected by missing values:")
print(missing_rows["Label"].value_counts())

Rows with missing values: 7

Classes affected by missing values:
Label
DoS Hulk    4
BENIGN      3
Name: count, dtype: int64


In [103]:
# Drop rows with missing values
df_clean = df_clean.dropna().copy()

# Display the results of removing the missing values
print("Dataset after removing missing values:", df_clean.shape)
print("Remaining missing values:", df_clean.isna().sum().sum())

Dataset after removing missing values: (37301, 81)
Remaining missing values: 0


In [105]:
# Combine all DoS attack types into the attack category
df_clean["Label"] = df_clean["Label"].replace({"DoS GoldenEye": "attack", "DoS Hulk": "attack", "DoS slowhttptest": "attack", "DoS slowloris": "attack"})

# Rename BENIGN to benign
df_clean["Label"] = df_clean["Label"].replace({"BENIGN": "benign"})

In [111]:
# Convert labels into numerical values
df_clean["Label"] = df_clean["Label"].map({"attack": 1,"benign": 0})

# Display the class distribution
print(df_clean["Label"].value_counts())

Label
1    32347
0     4954
Name: count, dtype: int64


In [107]:
# Display the final dataset dimensions
print("Final dataset shape:", df_clean.shape)

# Display the first five rows
display(df_clean.head())

Final dataset shape: (37301, 81)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,source_file,target
0,443,300491,7,5,376,3508,201,0,53.714286,78.686539,...,0.0,0,0,0.0,0.0,0,0,benign,ids_0.csv,0
1,37218,21,2,1,0,0,0,0,0.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,benign,ids_0.csv,0
2,53,176,2,2,148,212,74,74,74.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,benign,ids_0.csv,0
3,53,218,2,2,108,330,54,54,54.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,benign,ids_0.csv,0
4,53,30753,2,2,70,126,35,35,35.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,benign,ids_0.csv,0


In [112]:
# Save the cleaned dataset as a CSV file
df_clean.to_csv("IDS_cleaned.csv", index=False)

In [113]:
# Load the cleaned CSV file
IDS_cleaned_csv = pd.read_csv("IDS_cleaned.csv")

In [114]:
# Display the size of the dataset
print("Loaded dataset shape:", IDS_cleaned_csv.shape)

# Display the number of records in each class
print("\nClass distribution:")
print(IDS_cleaned_csv["Label"].value_counts())

# Display the first 20 rows
display(IDS_cleaned_csv.head(20))

Loaded dataset shape: (37301, 81)

Class distribution:
Label
1    32347
0     4954
Name: count, dtype: int64


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,source_file,target
0,443,300491,7,5,376,3508,201,0,53.714286,78.686539,...,0.0,0,0,0.0,0.0,0,0,0,ids_0.csv,0
1,37218,21,2,1,0,0,0,0,0.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,0,ids_0.csv,0
2,53,176,2,2,148,212,74,74,74.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,0,ids_0.csv,0
3,53,218,2,2,108,330,54,54,54.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,0,ids_0.csv,0
4,53,30753,2,2,70,126,35,35,35.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,0,ids_0.csv,0
5,443,125676,9,5,506,3743,204,0,56.222222,79.452467,...,0.0,0,0,0.0,0.0,0,0,0,ids_0.csv,0
6,123,16306,1,1,48,48,48,48,48.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,0,ids_0.csv,0
7,56434,2242,2,0,12,0,6,6,6.000000,0.000000,...,0.0,0,0,0.0,0.0,0,0,0,ids_0.csv,0
8,443,5659933,9,6,380,4091,187,0,42.222222,75.935462,...,0.0,256058,256058,5403872.0,0.0,5403872,5403872,0,ids_0.csv,0
9,39080,214,3,1,31,6,31,0,10.333333,17.897858,...,0.0,0,0,0.0,0.0,0,0,0,ids_0.csv,0
